# BUILDING A NEUTRAL NETWORK FROM SCRATCH
AUTHOR: TEMITOPE FAROTIMI

DATE: 03/27/2026

This is my first "learning by doing" project for my program for mastering Machine Learning. 

The task is to build from scratch a simple neural network that takes as input the data for three variables (features) and train it to predict another variable.

Specifically, the objective is to create a SimpleNeuralNet class that:
   1. Builds two layers of neurons (3 and 2 neurons respectively).
   2. Performs a forward pass of training data.
   3. Calculates the loss (Mean Squared Error)
   4. Performs Gradient Descent to minimize the loss.
   5. Performs backpropagation to tune the weights at each neuron.
   6. Validates the network using a hold-out dataset and provides performance metrics.

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

np.random.seed(42)

In [3]:
np.set_printoptions(precision=3, suppress=True, linewidth=120)

In [4]:
class SimpleNeuralNet():

    def __init__(self, dataset, target):
        assert dataset.shape[0] >= 30, "data train set must have >= 30 rows"

        self.target_name = target
        self.target = dataset[self.target_name]

        self.layers = {}
        self.outputs = {}
        self.activations = {}

        self.features = dataset.drop(columns=[self.target_name])
        self.feature_names = self.features.columns
        self.no_features = len(self.features.columns)

        self.train_X = self.features.iloc[:-10]
        self.test_X = self.features.iloc[-10:]
        self.train_Y = self.target.iloc[:-10]
        self.test_Y = self.target.iloc[-10:]

        self.n_batches = None
        self.n_epochs = 200
        self.n_learning_rate = 0.01
        self.debug_backprop = False  # flip to True to watch delta travel

        print("Dataset: ", dataset.shape)
        print("Train Set: ", self.train_X.shape)
        print("Test Set: ", self.test_X.shape)

    def initialize_neurons(self, n_neurons: list):
        self.n_neurons = n_neurons
        self.last_layer = len(n_neurons)

        for i, n in enumerate(n_neurons):
            fan_in = self.no_features if i == 0 else n_neurons[i - 1]
            self.layers[i + 1] = np.random.randn(fan_in, n) * np.sqrt(1 / fan_in)
            print(i + 1, self.layers[i + 1].shape)

        return self.layers

    def relu_activation(self, X) -> np.ndarray:
        return np.maximum(0, X)

    def _process_batch(self):
        self.n_batches = int(len(self.train_X) / 10) + 1
        batch_mse = []

        for i in range(0, len(self.train_X), 10):
            self.batch = np.array(self.train_X.iloc[i:i + 10])
            self.activations[0] = self.batch

            for j, layer in self.layers.items():
                prev = self.activations[j - 1]
                self.outputs[j] = prev @ layer
                if j == self.last_layer:
                    self.activations[j] = self.outputs[j]
                else:
                    self.activations[j] = self.relu_activation(self.outputs[j])

            self.final = self.activations[self.last_layer]
            self.batch_train = np.array(self.train_Y[i:i + 10]).reshape(-1, 1)
            self.y_preds = np.array(self.final).reshape(-1, 1)

            mse = ((self.batch_train - self.y_preds) ** 2).mean()
            batch_mse.append(float(mse))

            self.batch_results = {
                'y_preds': self.y_preds,
                'batch_train': self.batch_train,
                'mse': mse,
            }

            self.back_prop(self.y_preds, self.batch_train)

        return self.batch_results

    def forward_pass(self):
        loss_array = []
        for i in range(0, self.n_epochs):
            results = self._process_batch()
            self.batch_loss = results['mse']

            if i % 10 == 0:
                print(f"Loss after {i} epochs: {self.batch_loss:.4f}")

            loss_array.append(self.batch_loss)

        return plt.plot(loss_array)

    def back_prop(self, output, batch_y):
        delta = 2 * (output - batch_y) / len(batch_y)

        if self.debug_backprop:
            print(f"\nstart: delta from loss has shape {delta.shape}")

        for j in reversed(list(self.layers.keys())):
            if self.debug_backprop:
                print(f"\n--- layer {j} ---")
                print(f"  delta arrived with shape {delta.shape}")

            if j != self.last_layer:
                delta = delta * (self.outputs[j] > 0)
                if self.debug_backprop:
                    print(f"  after ReLU mask:        {delta.shape}")

            grad_W = self.activations[j - 1].T @ delta
            if self.debug_backprop:
                print(f"  grad_W = activations[{j-1}].T @ delta  ->  {self.activations[j-1].T.shape} @ {delta.shape} = {grad_W.shape}")
                print(f"  (layers[{j}].shape is        {self.layers[j].shape}  -- must match grad_W)")

            delta = delta @ self.layers[j].T
            if self.debug_backprop:
                print(f"  delta @ layers[{j}].T   ->  new delta {delta.shape}  (ready for layer {j-1})")

            self.layers[j] -= self.n_learning_rate * grad_W


In [5]:
from sklearn.datasets import load_diabetes
import pandas as pd

# This dataset tracks disease progression based on age, bmi, bp, etc.
data = load_diabetes()
df = pd.DataFrame(data.data, columns=data.feature_names)
target = data.target # This is your 'y'
diabetes_df = pd.concat([df, pd.Series(target, name='target')], axis=1)

In [6]:
diabetes_df

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0
...,...,...,...,...,...,...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207,178.0
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018114,0.044485,104.0
439,0.041708,0.050680,-0.015906,0.017293,-0.037344,-0.013840,-0.024993,-0.011080,-0.046883,0.015491,132.0
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044529,-0.025930,220.0


In [ ]:
neural_net = SimpleNeuralNet(diabetes_df, "target")

layers = neural_net.initialize_neurons([10, 8, 6,5,2,1])

test = neural_net.forward_pass()

Dataset:  (442, 11)
Train Set:  (432, 10)
Test Set:  (10, 10)
1 (10, 10)
2 (10, 8)
3 (8, 6)
4 (6, 5)
5 (5, 2)
6 (2, 1)


NameError: name 'test_nn' is not defined